### 0. Setup

#### 0.1 Importación de librerías necesarias

In [3]:
!pip install SpeechRecognition pydub
!pip install langdetect
!pip install openai-whisper
!pip install torch
!pip install pyaudio
!pip install ollama
!pip install gtts
!pip install pygame
!pip install playsound==1.2.2
!pip install edge_tts
!pip install deep-translator
!pip install easyocr
!pip install transformers
!pip install -U bitsandbytes
import speech_recognition as sr
import whisper as w
import torch
from langdetect import detect
import re
import ollama
from pydub import AudioSegment
import os
from gtts import gTTS
import pygame
from playsound import playsound
import edge_tts
import asyncio
import json
import tkinter as tk
from tkinter import filedialog
from transformers import CLIPProcessor, CLIPModel, Qwen2VLForConditionalGeneration, AutoProcessor, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from PIL import Image
import os.path
import torch
import pandas as pd
from deep_translator import GoogleTranslator
import easyocr
import cv2
import subprocess

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=c42a244d1ea3b3739833aad5d1925fa86e4f0afbbc741e279d19c8f1b486570e
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 54.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=7917ad357ecd632cf1ac37a4f33fd2cdb4dd9bf3157df3d1c00a3bc5a53f1fdb
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successful

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


#### 0.2 Configuración del modelo T2T dependiendo de dónde se ejecute el código (colab o local)

In [4]:
# Modelo chatbot a utilizar
MODELO_CHAT = "microsoft/phi-3-mini-4k-instruct"

### 1. Input

#### 1.1. S2T

In [5]:
def audio_a_wav(audio):
    """
    Esta función devuelve un archivo de audio en formato .wav.
    Argurmento:
    - audio: archivo de audio en cualquier formato.
    """
    nombre, extension = os.path.splitext(audio)
    extension = extension.lower()

    if extension == ".wav":
        return audio
    else:
        print("Convirtiendo el archivo a .wav.")
        archivo_nuevo = nombre + "convertido.wav"
        try:
            audio = AudioSegment.from_file(audio)
            audio.export(archivo_nuevo, format="wav")
            print("Archivo convertido a .wav.")
            return archivo_nuevo
        except:
            print("Error al convertir el archivo a .wav.")
            return None

In [6]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Modelo base
model = w.load_model("base", device=device)

def input_speech(audio = None):
    """
    Esta función transcribe un audio en formato .wav a texto o activa un micrófono en directo para guardar temporalmente el audio. Después de obtener el audio, lo transcribe y devuelve la transcripción y el idioma.
    Argumento:
    - audio: archivo de audio en formatio .wav (por defecto, audio = None).
    """
    # Opción 1: Se le pasa un archivo de audio .wav
    if audio != None:
        audio_a_wav(audio)
        # Utilizamos whisper porque procesa el archivo directamente, detecta el idioma y lo transcribe (con speech_recognition había que especificar el idioma previamente)
        result = model.transcribe(audio)
        # Se devuelve la transcripción del audio y el idioma
        return result["text"].strip(), result["language"]

    # Opción 2: Audio en directo
    else:
        r = sr.Recognizer()
        with sr.Microphone() as source:
            # Para reconocer el nivel de ruido en el audio y eliminar esa parte
            r.adjust_for_ambient_noise(source, duration=1)
            r.pause_threshold = 2.0
            # Se escucha el audio
            audio_data = r.listen(source, phrase_time_limit=20)

            # Guardamos el audio temporalmente ya que whisper lo necesita
            with open("audio_temporal.wav", "wb") as a:
                a.write(audio_data.get_wav_data())

            result = model.transcribe("audio_temporal.wav")
            # Devolvemos el texto y el idioma
            return result["text"].strip(), result["language"]

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 144MiB/s]


#### 1.2. T2T

In [7]:
def input_text():
    """
    Esta función pide un texto al usuario, detectando el idioma. Devuelve el texto escrito y el idioma.
    """
    # El usuario escribe sobre qué quiere información
    texto = input("¿Sobre qué quieres información?")

    # Detectamos el idioma
    try:
        idioma = detect(texto)
    except:
        # Idioma por defecto
        idioma = "es"

    print(f"Idioma del texto: {idioma}")
    # Devolvemos el texto y el idioma
    return texto, idioma

#### 1.3. Cleaning text

In [8]:
def cleaning_text(texto, idioma):
    """
    Esta función limpia un texto, para mejorar la legibilidad para después pasarselo a la IA. Devuelve el texto corregido.
    Argumentos:
    - texto: texto a limpiar.
    - idioma: idioma en el que está el texto.
    """
    # strip para eliminar espacios en blanco
    # capitalize para poner la primera letra en mayúscula
    texto_limpio = texto.strip().capitalize()

    # Muletillas a eliminar (español e inglés)
    muletillas = {
        "es": [r"\btipo+\b", r"\beh+\b", r"\besto+\b", r"\bpues+\b", r"\ben\s+plan\b"],
        "en": [r"\bum+\b", r"\beh+\b", r"\blike\b", r"\byou know\b"]
    }

    # Eliminamos las muletillas
    if idioma in muletillas:
        for i in muletillas[idioma]:
            texto_limpio = re.sub(i, "", texto_limpio, flags = re.IGNORECASE)

    # Eliminamos los n espacios que se han creado al eliminar las muletillas y lo sustitumos por un espacio
    texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()

    return texto_limpio

### 2. T2T

In [9]:
def respuesta_texto(texto_limpio, idioma, opciones, lugar):
    """
    Esta función devuelve una respuesta por la IA sobre x tema dado por el usuario.
    Argumentos:
    - texto_limpio: texto dado por el usuario y pasado por cleaning_text.
    - idioma: idioma detectado.
    - opciones:

    """
    # Interfaz
    print("========================================================")
    print(f"{opciones['titulo']}")
    print("========================================================")
    print(f"{opciones['pregunta']}")
    print(opciones['opcion_1'])
    print(opciones['opcion_2'])
    print(opciones['opcion_3'])


    op = input(f"\n")

    estilo_instruccion = opciones['instrucciones_ia'].get(op, opciones['instrucciones_ia']["1"])

    # Qué queremos que nos devuelva la IA
    prompt = f"""
    Eres un guía turístico experto.
    Tu objetivo es hablar y contar la historia sobre: {texto_limpio}
    Como contexto: El viaje realizado es a {lugar}, pero tu respuesta debe centrarse principalmente en: {texto_limpio},
    Instrucción específica: {estilo_instruccion}
    Ten en cuenta que la respuesta va a ser escuchada en voz alta. Por tanto, no escribas caracteres, emoticonos. Solo necesito el texto con la respuesta.
    IMPORTANTE: Debes responder OBLIGATORIAMENTE en el idioma correspondiente al código ISO: {idioma}.
    """

    # Cargamos el modelo
    tokenizer = AutoTokenizer.from_pretrained(MODELO_CHAT)
    model = AutoModelForCausalLM.from_pretrained(
        MODELO_CHAT,
        dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa"
    )

    # Preprocesado de la entrada
    messages = [
            {"role": "user", "content": prompt}
        ]
    input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    prompt_tokens = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Generamos la respuesta
    outputs = model.generate(**prompt_tokens, max_new_tokens=1024, cache_implementation="static")
    # Se decodifica
    respuesta_ia = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Devolvemos la respuesta generada
    return respuesta_ia

### 3. I2T

In [10]:
def info_con_imagen(idioma, lugar_viaje):
    # Obtener imagen del usuario
    ruta = cargar_imagen_tk()
    imagen = Image.open(ruta)

    # Cargar modelo Qwen2 para VQA
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
    )
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

    # Pregunta (identificación monumento)
    pregunta = "Which cultural site, LOCATED IN " + lugar_viaje.upper() + " is in this image?"
    pregunta = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": pregunta}
        ]
    }]

    # Preprocesar pregunta
    text_prompt = processor.apply_chat_template(pregunta, add_generation_prompt=True)
    inputs = processor(
        text=[text_prompt], images=[imagen], padding=True, return_tensors="pt"
    )
    inputs = inputs.to("cuda")

    # Generación de la respuesta
    output_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(inputs.input_ids, output_ids)
    ]
    respuesta = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )

    # Output
    print(respuesta)

    # Por si después se quiere más información, devolvemos la respuesta para un futuro prompt
    return respuesta

In [11]:
# PRUEBA
# respuesta_texto("información sobre la sagrada familia", "es")

### 3. Leer y traducir texto de una imagen

In [12]:
def leer_imagen(ruta_imagen, idioma=['en']):
  # Cargar imagen
  image = cv2.imread(ruta_imagen)

  # Cargar lector
  reader = easyocr.Reader(idioma)

  # Leer texto
  lectura = reader.readtext(image, paragraph=True, detail=0)
  return lectura


def cargar_imagen_tk():
    # Ventana, escondida
    root = tk.Tk()
    root.withdraw()

    # Para que sea más cómodo lo dejo donde tengo las imágenes
    carpeta_inicial = "imagenes"

    ruta = filedialog.askopenfilename(
        initialdir=carpeta_inicial,
        title="Selecciona una imagen",
        filetypes=[
            ("PNG files", "*.png"),
            ("JPEG files", "*.jpg"),
            ("JPEG files", "*.jpeg"),
            ("BMP files", "*.bmp")
        ]
    )
    return ruta


def menu_idiomas(texto_opciones, texto_despedidas, texto_traduciendo, idioma):
  ruta = cargar_imagen_tk()
  print(texto_opciones, flush=True)

  opcion = "aaaa"
  while opcion not in ["1","2","3","4","5","6","7"]:
    opcion = input("\nSelecciona una opción (1, 2, 3, 4, 5, 6 o 7): ")
    if opcion == "1":
      texto = leer_imagen(ruta, ['en'])
    if opcion == "2":
      texto = leer_imagen(ruta, ['ru','en'])
    if opcion == "3":
      texto = leer_imagen(ruta, ['ch_tra','en'])
    if opcion == "4":
      texto = leer_imagen(ruta, ['ch_sim', 'en'])
    if opcion == "5":
      texto = leer_imagen(ruta, ['ja', 'en'])
    if opcion == "6":
      texto = leer_imagen(ruta, ['ko','en'])
    if opcion == "7":
      return texto_despedidas

  texto_junto = " \n".join(texto)
  # Limpiar el texto a traducir
  texto_junto = re.sub('[$><|+`~]', '', texto_junto)
  print(texto_traduciendo)
  texto_traducido = GoogleTranslator(source='auto', target=idioma).translate(texto_junto)

  return texto_traducido

### 4. Identificación de un monumento con una foto

In [13]:
# def calcular_embeddings_monumentos():
#     """
#     Leer todos los nombres de los lugares de valor de la lista de patrimonio mundial de
#     la unesco, se guardan los embeddings de estos lugares para poder identificarlos.
#     Los nombres se obtienen de la base de datos disponible en:
#     https://data.unesco.org/explore/dataset/whc001/information/?flg=es-es&sort=date_inscribed
#     """
#     datos_patrimonio = pd.read_csv("datos/whc001.csv")

#     titulos = ["a photo of the " + nombre for nombre in datos_patrimonio['Name EN'].tolist()]

#     # Cargar modelo
#     model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
#     processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

#     caracteristicas = []
#     # Procesar el texto poco a poco, si no es muy exigente con la RAM
#     for texto in titulos:
#         text_inputs = processor(text=texto, return_tensors="pt", padding=True)
#         with torch.no_grad(): # que no guarde gradientes (menos ram)
#             text_features = model.get_text_features(**text_inputs)
#             text_features = text_features.pooler_output
#         caracteristicas.append(text_features.squeeze(0))

#     tensor_unico = torch.stack(caracteristicas)

#     # Normalizar las características
#     tensor_unico = tensor_unico / tensor_unico.norm(dim=-1, keepdim=True)

#     # Guardamos las características
#     torch.save({"titulos": datos_patrimonio['Name EN'].tolist(),
#                 "caracteristicas": tensor_unico}, "datos/preproc_monumentos.pt")

# def identificar_monumento(imagen):
#     """
#     Identificar monumento comparando los embeddings de cada texto con las
#     características de las imágenes, seleccionando mayor similitud
#     """
#     # Si no tenemos ya el archivo con las características de
#     # los monumentos, calcularlos
#     if not os.path.isfile("datos/preproc_monumentos.pt"):
#         calcular_embeddings_monumentos()

#     # Modelo que extrae características de la imagen a comparar con texto
#     model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
#     processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
#     model.eval()

#     # Embeddings de cada texto de los monumentos conocidos, junto con su nombre
#     datos = torch.load("datos/preproc_monumentos.pt", weights_only=False)
#     titulos = datos["titulos"]
#     text_embeddings = datos["caracteristicas"]

#     # Extracción características de la imagen
#     input = processor(images=imagen, return_tensors="pt")
#     with torch.no_grad():
#         image_features = model.get_image_features(**input)
#         image_features = image_features.pooler_output
#     image_features = image_features / image_features.norm(dim=-1, keepdim=True)

#     # Similitud con los textos
#     similarity = image_features @ text_embeddings.T

#     # Título más similar a la imagen
#     best_idx = similarity.argmax().item()
#     return titulos[best_idx]

#     # 5 más similares, para evitar posibles errores
#     # values, indices = similarity.topk(5, dim=-1)
#     # values = values[0]
#     # indices = indices[0]

#     # print(values)
#     # for i in indices:
#     #     print(titulos[i.item()])

# def info_monumento(nombre_monumento, idioma, lugar_viaje):
#     # Qué queremos que nos devuelva la IA
#     prompt = f"""
#     Eres un guía turístico experto.
#     Tu objetivo es hablar y contar la historia sobre: {nombre_monumento},
#     Como contexto: El viaje realizado es a {lugar_viaje}, pero tu respuesta debe centrarse principalmente en: {nombre_monumento},
#     Instrucción específica: Hazme una breve introducción a este patrimonio, como contexto recuerda que se encuentra en {lugar_viaje}.
#     Ten en cuenta que la respuesta va a ser escuchada en voz alta. Por tanto, no escribas caracteres, emoticonos, ni asteriscos *. Solo necesito el texto con la respuesta.
#     IMPORTANTE: Debes responder OBLIGATORIAMENTE en el idioma correspondiente al código ISO: {idioma}.
#     """

#     # Llamamos a ollama
#     respuesta_ia = ollama.chat(
#         model=MODELO_CHAT,
#         messages=[{'role': 'user', 'content': prompt}]
#     )

#     # Devolvemos la respuesta generada
#     return respuesta_ia['message']['content']

# def info_con_imagen(idioma, lugar_viaje):
#     """
#     Esta función permite subir una imagen, detectar a qué patrimonio de la humanidad
#     corresponde y dar una breve introducción acerca del monumento.
#     """
#     ruta_imagen = cargar_imagen_tk()
#     imagen = Image.open(ruta_imagen)
#     monumento = identificar_monumento(imagen)
#     print(info_monumento(monumento, idioma, lugar_viaje))

#     return monumento

In [14]:

#identificar_monumento(imagen)

In [15]:
#calcular_embeddings_monumentos()

#### 4.1 Evaluar el modelo de identificación de monumentos

In [16]:
# from transformers import logging
# logging.set_verbosity_error()
# logging.disable_progress_bar()

# def identificar_monumento_CLIP(imagen):
#     """
#     Identificar monumento comparando los embeddings de cada texto con las
#     características de las imágenes, seleccionando mayor similitud
#     """
#     # Si no tenemos ya el archivo con las características de
#     # los monumentos, calcularlos
#     if not os.path.isfile("datos/preproc_monumentos.pt"):
#         calcular_embeddings_monumentos()

#     # Modelo que extrae características de la imagen a comparar con texto
#     model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", ignore_mismatched_sizes=True)
#     processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
#     model.eval()

#     # Embeddings de cada texto de los monumentos conocidos, junto con su nombre
#     datos = torch.load("datos/preproc_monumentos.pt", weights_only=False)
#     titulos = datos["titulos"]
#     text_embeddings = datos["caracteristicas"]

#     # Extracción características de la imagen
#     input = processor(images=imagen, return_tensors="pt")
#     with torch.no_grad():
#         image_features = model.get_image_features(**input)
#         image_features = image_features.pooler_output
#     image_features = image_features / image_features.norm(dim=-1, keepdim=True)

#     # Similitud con los textos
#     similarity = image_features @ text_embeddings.T

#     # # Título más similar a la imagen
#     # best_idx = similarity.argmax().item()
#     # return titulos[best_idx]

#     # 40 más similares, para evitar posibles errores
#     values, indices = similarity.topk(40, dim=-1)
#     values = values[0]
#     indices = indices[0]
#     nombres40 = []
#     for i in indices:
#         nombres40.append(titulos[i.item()])
#     # Juntamos todo, esta string se pasará a un prompt por lo que usamos || para luego indicar cómo se separa
#     nombres40 = '||'.join(nombres40)
#     return nombres40

# def respuesta_mejora_CLIP(lista_posibilidades, lugar, idioma):
#     # Qué queremos que nos devuelva la IA
#     # prompt = f"""
#     # Eres un guía turístico experto.
#     # Di cuál es el primer lugar de interés en la siguiente lista de elementos separados por ||: {lista_posibilidades} que se encuentra en {lugar},
#     # una vez identificado solo este elemento de la lista debe aparecer en tu respuesta.
#     # Tu objetivo es hablar y contar la historia sobre el lugar de interés identificado,
#     # Como contexto: El viaje realizado es a {lugar}, pero tu respuesta debe centrarse principalmente en el lugar de interés,
#     # Instrucción específica: Hazme una breve introducción a este patrimonio, como contexto recuerda que se encuentra en {lugar}.
#     # Ten en cuenta que la respuesta va a ser escuchada en voz alta. Por tanto, no escribas caracteres, emoticonos, ni utilices letra en cursiva o negrita. Solo necesito el texto con la respuesta.
#     # IMPORTANTE: Debes responder OBLIGATORIAMENTE en el idioma correspondiente al código ISO: {idioma}.
#     # """
#     prompt = f"""
#     You are an expert tour guide.
#     You must find the first cultural site on this list: {lista_posibilidades} that is LOCATED IN {lugar.upper()}, your response should only mention this cultural site and no other elements of the list.
#     Your objective is speaking and explaining the history behind the cultural site.
#     As context: remember this is LOCATED IN {lugar.upper()}, though your response should be focused on the cultural site.
#     Specific instruction: Make a brief introduction to this cultural site, keep in mind that your response will be heard aloud so you must not
#     include any special characters, emoticons or use bold or italic letters.
#     IMPORTANT: You are OBLIGATED TO answer in the language corresponding to the ISO code: {idioma}.
#     """

#     # Llamamos a ollama
#     respuesta_ia = ollama.chat(
#         model=MODELO_CHAT,
#         messages=[{'role': 'user', 'content': prompt}]
#     )

#     # Devolvemos la respuesta generada
#     return respuesta_ia['message']['content']

# def info_con_imagen(idioma, lugar_viaje):
#     """
#     Esta función permite subir una imagen, detectar a qué patrimonio de la humanidad
#     corresponde y dar una breve introducción acerca del monumento.
#     """
#     ruta_imagen = cargar_imagen_tk()
#     imagen = Image.open(ruta_imagen)
#     posibilidades = identificar_monumento_CLIP(imagen)
#     print(respuesta_mejora_CLIP(posibilidades, idioma, lugar_viaje))

In [17]:
# datos_patrimonio = pd.read_csv("datos/whc001.csv")

# links_imagenes = datos_patrimonio['Main Image'].tolist()
# nombres = datos_patrimonio['Name EN'].tolist()
# for i in range(30):
#     print(nombres[i])
#     print(links_imagenes[i])

In [18]:
# carpeta = "imagenes/imagenes_patrimonio"

# rutas_imagenes = [
#     os.path.join(carpeta, f)
#     for f in os.listdir(carpeta)
#     if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".gif"))
# ]

# for ruta in rutas_imagenes:
#     print(ruta)
#     img = Image.open(ruta)
#     ident = identificar_monumento_CLIP(img)
#     print(ident)
#     print("\n")

In [19]:
# Con contexto adicional: país
# lugares = ['Albania', 'Australia', 'Angola', 'Australia', 'Argelia', 'Armenia', 'Argentina', 'Algeria', 'Armenia', 'Argelia', 'Argentina', 'Argelia', 'Australia']

# carpeta = "imagenes/imagenes_patrimonio"

# rutas_imagenes = [
#     os.path.join(carpeta, f)
#     for f in os.listdir(carpeta)
#     if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".gif"))
# ]

# for i, ruta in enumerate(rutas_imagenes):
#     print(ruta)
#     img = Image.open(ruta)
#     ident = monumento_CLIP_mas_contexto(img, lugares[i])
#     print(ident)
#     print("\n")

In [20]:
# ruta = "imagenes/prueba_leer.png"
# type(Image.open(ruta))

### 4. Output (2. T2T o T2S)

In [21]:
#### VOZ MUY ROBOTIZADA
# def respuesta_audio(respuesta_ia, idioma):
#     voz = gTTS(text = respuesta_ia, lang = idioma)

#     voz_save = "respuesta_audio.mp3"
#     voz.save(voz_save)

#     pygame.mixer.init()
#     pygame.mixer.music.load(voz_save)
#     pygame.mixer.music.play()

#     while pygame.mixer.music.get_busy():
#         continue


In [22]:
async def respuesta_audio(respuesta_ia, idioma):
    """
    Esta función devuelve una respuesta por voz en un idioma dado (la respuesta es la que ha dado la IA, la pasamos a audio).
    Argumentos:
    - respuesta_ia: texto que ha dado la IA.
    - idioma: idioma en qué queremos la voz.
    """
    voces = await edge_tts.list_voices()
    voz_encontrada = None

    # Por defecto escogemos que el idioma "es" es "es-ES"
    if idioma == "es":
        for voz in voces:
            if "es-ES" in voz["ShortName"]:
                voz_encontrada = voz["ShortName"]
                break

    # Encontramos el idioma en el que queremos la respuesta
    if not voz_encontrada:
        for v in voces:
            if v["ShortName"].startswith(idioma):
                voz_encontrada = v["ShortName"]
                break

    # Si no encuentra la voz sobre ese idioma, por defecto usamos este
    if not voz_encontrada:
        voz_encontrada = "en-US-GuyNeural"

    # Guardamos temporalmente el audio
    archivo = "respuesta_voz.mp3"
    communicate = edge_tts.Communicate(respuesta_ia, voz_encontrada)
    await communicate.save(archivo)

    os.system(f"afplay {archivo}")
    os.remove(archivo)

In [23]:
# PRUEBA
# texto_ia = respuesta_texto("información sobre la sagrada familia", "es")
# await respuesta_audio(texto_ia, "es-ES")

In [24]:
# def menu_principal():
#     print("\n--- Guía turística---")
#     print("1. Escribir")
#     print("2. Hablar/Audio")
#     print("3. Hacer foto")
#     print("4. Salir")

#     opcion = input("\nSelecciona una opción (1, 2, 3 o 4): ")

#     texto_sucio, idioma = None, None
#     if opcion == "1":
#         texto_sucio, idioma = input_text()

#     elif opcion == "2":
#         print("\n¿Quieres (a) hablar o (b) pasar un archivo .wav?")
#         op = input("Selecciona una opción (a o b): ").lower()
#         if op == "a":
#             print("Escuchando...")
#             texto_sucio, idioma = input_speech()
#         else:
#             ruta = input("Introduce la ruta .wav: ")
#             texto_sucio, idioma = input_speech(audio=ruta)

#     elif opcion == "3":
#         return

#     elif opcion == "4":
#         print("...")
#         return
#     else:
#         print("Opción no válida.")
#         return

#     if texto_sucio and idioma:
#         texto_limpio = cleaning_text(texto_sucio, idioma)
#         print(f"\n Procesando texto: {texto_limpio}")

#         respuesta_final = respuesta_texto(texto_limpio, idioma)

#         print("\n Respuesta:")
#         print(respuesta_final)
#         return respuesta_final

In [25]:
async def menu_principal():
    """
    Función para determinar en qué idioma quiere que esté la app el usuario (se lo pasamos a la IA para que lo traduzca).
    Se puede decir el idioma por voz o escribiéndolo.
    Se devuelve el código ISO del idioma.
    """
    # Qué idioma quiere el usuario?
    print("========================================================")
    print("Configuración del idioma")
    print("========================================================")
    print("Escribe el idioma (ej: 'español', 'english', 'italiano') o pulsa ENTER para decir el idioma con la voz.")

    op_idioma = input("\n")

    if op_idioma == "":
        print("Escuchando...")
        # Usamos Whisper para detectar qué idioma quiere el usuario
        texto, cod_idioma = input_speech()
        print(f"Idioma detectado: {texto} ({cod_idioma})")
    else:
        # También puede escribir el idioma que quiere
        prompt_iso = f"Dame solo el código ISO 639-1 (2 letras) para el idioma: {op_idioma}. Ejemplo: es, en, fr. No escribas nada más."

        # Cargamos el modelo
        tokenizer = AutoTokenizer.from_pretrained(MODELO_CHAT)
        model = AutoModelForCausalLM.from_pretrained(
            MODELO_CHAT,
            dtype=torch.bfloat16,
            device_map="auto",
            attn_implementation="sdpa"
        )
        # Preprocesado de la entrada
        messages = [
            {"role": "user", "content": prompt_iso}
        ]
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompt_tokens = tokenizer(input_text, return_tensors="pt").to(model.device)

        # Generamos la respuesta
        outputs = model.generate(**prompt_tokens, max_new_tokens=10, cache_implementation="static")
        # Se decodifica
        res = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print('RES')
        print(res.strip().lower())

        cod_idioma = res.strip().lower()
        print(f"Idioma detectado: {op_idioma} ({cod_idioma}). Traduciendo al idioma deseado.")

    # Devolvemos el código del idioma
    return cod_idioma

In [26]:
def comprobar_modelo_instalado(modelo):
    try:
        prueba = ollama.chat(model=modelo, messages = [{'role': 'user',
                                                        'content': 'Responde SOLAMENTE ¡Todo listo!'}])
    except:
        subprocess.run(["ollama", "pull", modelo], check=False)

In [27]:
# Menú a traducir según el idioma deseado por el usuario
MENU_COMPLETO = {
    "menu": {
        "bienvenida": "Menú principal",
        "opciones": [
            "1. Escribir una pregunta (Texto)",
            "2. Hablar con el guía o subir un audio (Audio)",
            "3. Hacer foto a un monumento",
            "4. Traducir texto de una foto",
            "5. Salir del menú"
        ],
        "instruccion": "Seleccione una opción (1, 2, 3, 4 o 5):",
        "intro": " ESCRIBIR INFORMACIÓN SOBRE QUÉ ES ESTA APP"
    },
    "mensajes": {
        "pregunta_texto": "A continuación escriba su duda o el tema sobre el que desea información.",
        "pregunta_audio_modo": "¿Quiere (a) hablar o (b) subir un archivo?",
        "instruccion_audio": "Selecciona una opción (a o b):",
        "escuchando": "Escuchando...",
        "ruta_archivo": "Introduzca la ruta del archivo .wav:",
        "procesando": "Procesando información. Espere unos segundos.",
        "foto_monumento": "Este servicio permite identificar por imágenes monumentos declarados patrimonio de la humanidad.",
        "foto": "Para comenzar, suba una imagen.",
        "mas_info_foto": "¿Quiere más información sobre este monumento? pulse (a) para seguir y cualquier otra tecla para salir",
        "traduciendo": "Traduciendo texto...",
        "error_opcion": "Opción no válida. Inténtelo de nuevo.",
        "despedida": "¡Adiós!"
    },
    "estilos": {
        "titulo": "Opciones de estilo",
        "opcion_1": "1. Resumen breve",
        "opcion_2": "2. Historia detallada",
        "opcion_3": "3. Datos curiosos",
        "pregunta": "Selecciona el tipo de información deseada (1, 2 o 3):",
        "instrucciones_ia": {
            "1": "Hazme un resumen breve.",
            "2": "Hazme una explicación detallada y técnica.",
            "3": "Cuéntame curiosidades."
        }
    },
    "contexto": {
        "lugar": "¿A dónde es tu viaje?"
    },
    "opciones_alfabeto":
        """¿Qué alfabeto utiliza el texto a traducir?
            1. Latino
            2. Cirílico
            3. Chino tradicional
            4. Chino simplificado
            5. Japonés
            6. Coreano
            7. Salir
            """
}


async def menu_traducido(cod_idioma):
    """
    Función para traducir MENU_COMPLETO al idioma deseado.
    Argumento:
    - cod_idioma: código del idioma al que se va a traducir toda la app.
    """
    # Si el idioma es español la traducción no es necesaria
    if cod_idioma=='es':
        return MENU_COMPLETO

    # prompt para ollama apra traducir MENU_COMPLETO
    prompt = f"""
    Traduce este JSON al idioma '{cod_idioma}'.
    Mantén las claves originales y traduce solo los valores.
    Responde SOLO el JSON.
    JSON: {json.dumps(MENU_COMPLETO, ensure_ascii=False)}
    """

    # Cargamos el modelo
    tokenizer = AutoTokenizer.from_pretrained(MODELO_CHAT)
    model = AutoModelForCausalLM.from_pretrained(
        MODELO_CHAT,
        dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa"
    )

    # Preprocesado de la entrada
    messages = [
            {"role": "user", "content": prompt}
        ]
    input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    prompt_tokens = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Generamos la respuesta
    outputs = model.generate(**prompt_tokens, max_new_tokens=1024, cache_implementation="static")
    # Se decodifica
    res_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    try:
        return json.loads(res_text)
    except:
        return MENU_COMPLETO


async def app():
    """
    Función ("app") que une todas la funciones.
    """
    # Obtenemos el idioma deseado
    print("========================================================", flush=True)
    print("BIENVENIDO A TU GUÍA TURÍSTICA CON IA", flush=True)
    print("========================================================", flush=True)

    cod_idioma = await menu_principal()

    # Traducimos la interfaz del menú
    interfaz = await menu_traducido(cod_idioma)

    print("\n")
    print(interfaz['contexto']['lugar'], flush=True)
    lugar_viaje = input(f"\n ")

    print("\n")
    print(f"{interfaz['menu']['intro']}\n", flush=True)

    while True:

        print("========================================================")
        print(f"{interfaz['menu']['bienvenida']} ", flush=True)
        print("========================================================")

        print(f"{interfaz['menu']['instruccion']}", flush=True)

        for opt in interfaz['menu']['opciones']:
            print(opt)
        opcion = input(f"\n ")

        texto_sucio, idioma_final = None, cod_idioma

        # Opción para escribir
        if opcion == "1":
            print(f"\n{interfaz['mensajes']['pregunta_texto']}", flush=True)
            texto_sucio = input("\n")

        # Opción para hablar
        elif opcion == "2":
            print(f"\n{interfaz['mensajes']['pregunta_audio_modo']}", flush=True)
            print(f"{interfaz['mensajes']['instruccion_audio']} ", flush=True)
            sub_opcion = input("\n").lower()

            # Hablando
            if sub_opcion == "a":
                print(interfaz['mensajes']['escuchando'], flush=True)
                texto_sucio, idioma_final = input_speech()
            # Subiendo un audio
            else:
                ruta = input(f"{interfaz['mensajes']['ruta_archivo']} ", flush=True)
                texto_sucio, idioma_final = input_speech(audio=ruta)

        # Hacer foto
        elif opcion == "3":
            print(interfaz['mensajes']['foto_monumento'], flush=True)
            print(interfaz['mensajes']['foto'], flush=True)
            monumento = info_con_imagen(idioma_final, lugar_viaje)

            print(interfaz['mensajes']['mas_info_foto'], flush=True)
            sub_opcion = input("\n").lower()
            # Si se quiere más información acerca del monumento
            if sub_opcion == "a":
                texto_sucio = "Información acerca de " + monumento

        # Traducir texto de una imagen
        elif opcion == "4":
            print(interfaz['mensajes']['foto'], flush=True)
            print(menu_idiomas(interfaz['opciones_alfabeto'],
                               interfaz['mensajes']['despedida'],
                               interfaz['mensajes']['traduciendo'],
                               idioma_final))

        # Salida
        elif opcion == "5":
            print(interfaz['mensajes']['despedida'], flush=True)
            break

        # Error
        else:
            print(interfaz['mensajes']['error_opcion'], flush=True)
            continue

        if texto_sucio:
            print(f"\n{interfaz['mensajes']['procesando']}\n", flush=True)
            texto_limpio = cleaning_text(texto_sucio, idioma_final)
            respuesta = respuesta_texto(texto_limpio, idioma_final, interfaz['estilos'], lugar_viaje)
            print(f"\nRespuesta:\n{respuesta}", flush=True)

            print(f"\n¿Desea escuchar la respuesta? (s/n)", flush=True)
            audio_si_no = input("\n").lower().strip()

            if audio_si_no == "s":
                print("Generando audio. Espere unos segundos.", flush=True)
                await respuesta_audio(respuesta, idioma_final)

            print("Pulse ENTER para continuar.", flush=True)
            input("\n")


In [34]:
await menu_principal()

Configuración del idioma
Escribe el idioma (ej: 'español', 'english', 'italiano') o pulsa ENTER para decir el idioma con la voz.

español


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

W0324 18:15:02.257000 15912 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist

RES
dame solo el código iso 639-1 (2 letras) para el idioma: español. ejemplo: es, en, fr. no escribas nada más. es
Idioma detectado: español (dame solo el código iso 639-1 (2 letras) para el idioma: español. ejemplo: es, en, fr. no escribas nada más. es). Traduciendo al idioma deseado.


/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


'dame solo el código iso 639-1 (2 letras) para el idioma: español. ejemplo: es, en, fr. no escribas nada más. es'

In [29]:
#comprobar_modelo_instalado(MODELO_CHAT)

In [30]:
#await app()

# Me acaba de decir que la ciudad de las artes y las ciencias está en barcelna
# Respuesta:
# ¡Claro! Me alegra poder hablar sobre la ciudad de las artes y las ciencias, ubicada en Barcelona, España. A continuación, te doy un resumen breve:

# La Ciudad de las Artes y las Ciencias (en catalán, Ciutat de les Arts i les Ciències) es un complejo arquitectónico y cultural situado en el distrito de l'Eixample, en el corazón de Barcelona. Fue diseñada por los arquitectos Félix Candela, Rafael Moneo y Santiago Calatrava, y se inauguró en 1990.

# El complejo consta de varios edificios y estructuras que reflejan la fusión de arte, ciencia y tecnología. Entre ellos destacan el Palau de les Arts Reina Sofía (teatro y centro de música), el Museo de las Ciencias Príncipe Felipe, la Torre l'Ascensor, el Hemisfèric (un planetario y observatorio) y la Ciudad de las Esculturas.

# La ciudad es un lugar perfecto para la contemplación, la educación y el entretenimiento. Ofrece visitas guiadas, exhibiciones temporales y permanentes, conciertos y espectáculos, lo que la convierte en uno de los atractivos turísticos más populares de Barcelona.

# ¡Espero que disfrutes visitando esta maravillosa ciudad!

In [31]:
#await app()

In [32]:
#ollama list

In [33]:
# ollama.chat(model='llama3', messages=[
#     {'role': 'user', 'content': 'Hola'}
# ])